In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [2]:
file_path = "../data/processed/hourly_zone_demand_2025_01.parquet"

demand = pd.read_parquet(file_path)

print("Shape:", demand.shape)

print("\nColumns:")
print(demand.columns.tolist())

print("\nDate range:")
print(demand["datetime"].min(), "to", demand["datetime"].max())

demand.head()

Shape: (195672, 13)

Columns:
['datetime', 'date', 'hour', 'day_of_week', 'day_of_week_num', 'day_of_month', 'month', 'is_weekend', 'PULocationID', 'Borough', 'Zone', 'service_zone', 'demand']

Date range:
2025-01-01 00:00:00 to 2025-01-31 23:00:00


,datetime,date,hour,day_of_week,day_of_week_num,day_of_month,month,is_weekend,PULocationID,Borough,Zone,service_zone,demand
0,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,1,EWR,Newark Airport,EWR,0
1,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,2,Queens,Jamaica Bay,Boro Zone,0
2,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,3,Bronx,Allerton/Pelham Gardens,Boro Zone,74
3,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,4,Manhattan,Alphabet City,Yellow Zone,242
4,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,5,Staten Island,Arden Heights,Boro Zone,18


Sort and verify time continuity

In [3]:
demand = demand.sort_values(
    ["PULocationID", "datetime"]
).reset_index(drop=True)

demand[["PULocationID", "datetime", "demand"]].head(10)

,PULocationID,datetime,demand
0,1,2025-01-01 00:00:00,0
1,1,2025-01-01 01:00:00,0
2,1,2025-01-01 02:00:00,0
3,1,2025-01-01 03:00:00,0
4,1,2025-01-01 04:00:00,0
5,1,2025-01-01 05:00:00,0
6,1,2025-01-01 06:00:00,0
7,1,2025-01-01 07:00:00,0
8,1,2025-01-01 08:00:00,0
9,1,2025-01-01 09:00:00,0


In [4]:
hours_per_zone = (
    demand.groupby("PULocationID")
          .size()
)

print("Minimum hours per zone:", hours_per_zone.min())
print("Maximum hours per zone:", hours_per_zone.max())
print("Expected hours:", 31 * 24)

print(
    "Zones with exactly 744 hours:",
    (hours_per_zone == 744).sum()
)

Minimum hours per zone: 744
Maximum hours per zone: 744
Expected hours: 744
Zones with exactly 744 hours: 263


In [5]:
time_gaps = (
    demand.groupby("PULocationID")["datetime"]
          .diff()
)

unexpected_gaps = time_gaps[
    time_gaps.notna() &
    (time_gaps != pd.Timedelta(hours=1))
]

print("Unexpected time gaps:", len(unexpected_gaps))

Unexpected time gaps: 0


Historical Demand Features

In [6]:
# Previous hour demand
demand["lag_1"] = (
    demand.groupby("PULocationID")["demand"]
          .shift(1)
)

# Demand two hours earlier
demand["lag_2"] = (
    demand.groupby("PULocationID")["demand"]
          .shift(2)
)

# Same hour on the previous day
demand["lag_24"] = (
    demand.groupby("PULocationID")["demand"]
          .shift(24)
)

# Same hour one week earlier
demand["lag_168"] = (
    demand.groupby("PULocationID")["demand"]
          .shift(168)
)

Rolling demand features

In [7]:
demand["rolling_mean_3"] = (
    demand.groupby("PULocationID")["demand"]
          .transform(
              lambda x: x.shift(1).rolling(3).mean()
          )
)

demand["rolling_mean_24"] = (
    demand.groupby("PULocationID")["demand"]
          .transform(
              lambda x: x.shift(1).rolling(24).mean()
          )
)

In [8]:
feature_columns = [
    "datetime",
    "PULocationID",
    "demand",
    "lag_1",
    "lag_2",
    "lag_24",
    "lag_168",
    "rolling_mean_3",
    "rolling_mean_24"
]

demand[feature_columns].head(175).tail(15)

,datetime,PULocationID,demand,lag_1,lag_2,lag_24,lag_168,rolling_mean_3,rolling_mean_24
160,2025-01-07 16:00:00,1,0,0.0,0.0,0.0,NaN,0.0,0.0
161,2025-01-07 17:00:00,1,0,0.0,0.0,0.0,NaN,0.0,0.0
162,2025-01-07 18:00:00,1,0,0.0,0.0,0.0,NaN,0.0,0.0
163,2025-01-07 19:00:00,1,0,0.0,0.0,0.0,NaN,0.0,0.0
164,2025-01-07 20:00:00,1,0,0.0,0.0,0.0,NaN,0.0,0.0
165,2025-01-07 21:00:00,1,0,0.0,0.0,0.0,NaN,0.0,0.0
166,2025-01-07 22:00:00,1,0,0.0,0.0,0.0,NaN,0.0,0.0
167,2025-01-07 23:00:00,1,0,0.0,0.0,0.0,NaN,0.0,0.0
168,2025-01-08 00:00:00,1,0,0.0,0.0,0.0,0.0,0.0,0.0
169,2025-01-08 01:00:00,1,0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
print("Missing values after feature engineering:")

print(
    demand[
        [
            "lag_1",
            "lag_2",
            "lag_24",
            "lag_168",
            "rolling_mean_3",
            "rolling_mean_24"
        ]
    ].isna().sum()
)

Missing values after feature engineering:
lag_1                263
lag_2                526
lag_24              6312
lag_168            44184
rolling_mean_3       789
rolling_mean_24     6312
dtype: int64


create the ML ready dataset

In [10]:
model_features = [
    "hour",
    "day_of_week_num",
    "day_of_month",
    "is_weekend",
    "PULocationID",
    "lag_1",
    "lag_2",
    "lag_24",
    "lag_168",
    "rolling_mean_3",
    "rolling_mean_24"
]

target = "demand"

model_data = demand[
    ["datetime"] + model_features + [target]
].dropna().copy()

print("Original rows:", len(demand))
print("ML-ready rows:", len(model_data))
print("Rows removed:", len(demand) - len(model_data))

print("\nML date range:")
print(model_data["datetime"].min(), "to", model_data["datetime"].max())

model_data.head()

Original rows: 195672
ML-ready rows: 151488
Rows removed: 44184

ML date range:
2025-01-08 00:00:00 to 2025-01-31 23:00:00


,datetime,hour,day_of_week_num,day_of_month,is_weekend,PULocationID,lag_1,lag_2,lag_24,lag_168,rolling_mean_3,rolling_mean_24,demand
168,2025-01-08 00:00:00,0,2,8,False,1,0.0,0.0,0.0,0.0,0.0,0.0,0
169,2025-01-08 01:00:00,1,2,8,False,1,0.0,0.0,0.0,0.0,0.0,0.0,0
170,2025-01-08 02:00:00,2,2,8,False,1,0.0,0.0,0.0,0.0,0.0,0.0,0
171,2025-01-08 03:00:00,3,2,8,False,1,0.0,0.0,0.0,0.0,0.0,0.0,0
172,2025-01-08 04:00:00,4,2,8,False,1,0.0,0.0,0.0,0.0,0.0,0.0,0


Chronological spli

In [11]:
train = model_data[
    model_data["datetime"] < "2025-01-25"
].copy()

validation = model_data[
    (model_data["datetime"] >= "2025-01-25") &
    (model_data["datetime"] < "2025-01-28")
].copy()

test = model_data[
    model_data["datetime"] >= "2025-01-28"
].copy()

print("TRAIN")
print("Rows:", len(train))
print("Range:", train["datetime"].min(), "to", train["datetime"].max())

print("\nVALIDATION")
print("Rows:", len(validation))
print("Range:", validation["datetime"].min(), "to", validation["datetime"].max())

print("\nTEST")
print("Rows:", len(test))
print("Range:", test["datetime"].min(), "to", test["datetime"].max())

TRAIN
Rows: 107304
Range: 2025-01-08 00:00:00 to 2025-01-24 23:00:00

VALIDATION
Rows: 18936
Range: 2025-01-25 00:00:00 to 2025-01-27 23:00:00

TEST
Rows: 25248
Range: 2025-01-28 00:00:00 to 2025-01-31 23:00:00


Build Baseline Models

In [13]:
import sklearn

print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.9.1


In [14]:
##Baseline 1: predict next-hour demand using the previous hour (lag_1).
###Baseline 2: predict demand using the same hour one week earlier (lag_168).
##We'll compare using:
#- MAE — average absolute prediction error
#- RMSE — penalizes large prediction errors more strongly 

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_val = validation["demand"]

# Baseline 1: previous hour
pred_lag1 = validation["lag_1"]

mae_lag1 = mean_absolute_error(y_val, pred_lag1)
rmse_lag1 = np.sqrt(mean_squared_error(y_val, pred_lag1))

# Baseline 2: same hour one week earlier
pred_lag168 = validation["lag_168"]

mae_lag168 = mean_absolute_error(y_val, pred_lag168)
rmse_lag168 = np.sqrt(mean_squared_error(y_val, pred_lag168))

print("BASELINE 1 — Previous Hour")
print(f"MAE:  {mae_lag1:.2f}")
print(f"RMSE: {rmse_lag1:.2f}")

print("\nBASELINE 2 — Same Hour Previous Week")
print(f"MAE:  {mae_lag168:.2f}")
print(f"RMSE: {rmse_lag168:.2f}")

BASELINE 1 — Previous Hour
MAE:  20.08
RMSE: 35.60

BASELINE 2 — Same Hour Previous Week
MAE:  19.84
RMSE: 40.37


In [15]:
pred_lag24 = validation["lag_24"]

mae_lag24 = mean_absolute_error(y_val, pred_lag24)
rmse_lag24 = np.sqrt(mean_squared_error(y_val, pred_lag24))

print("BASELINE 3 — Same Hour Previous Day")
print(f"MAE:  {mae_lag24:.2f}")
print(f"RMSE: {rmse_lag24:.2f}")

BASELINE 3 — Same Hour Previous Day
MAE:  37.68
RMSE: 75.77


In [16]:
output_path = "../data/processed/ml_ready_demand_2025_01.parquet"

model_data.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(model_data))

Saved: ../data/processed/ml_ready_demand_2025_01.parquet
Rows: 151488
